# The ESIS Level-4 Data Product

The Level-4 data product is the time-dependent spatial-spectral reconstruction
of an ESIS observation:
every frame of the Level-1 data, inverted with the multiplicative algebraic
reconstruction technique (MART) using the real, fitted instrument model as the
forward model.
Where the lower data levels live on the sensor grid, Level 4 lives on the
scene grid: the spectral radiance of the three brightest lines in the ESIS
passband — He I 584 Å, Mg X 610 Å, and O V 630 Å — each with a ±200 km/s
Doppler window, on a spatial grid matching the plate scale of the instrument.

The inversion pipeline is packaged in :meth:`esis.data.Level_4.from_level_1`:

1. Linearize the fitted optical system,
   :func:`esis.flights.f1.optics.distortion_fit`,
   with :meth:`optika.systems.SequentialSystem.linearize`,
   and cache the linearization and its conservative regridding weights.
2. Clip negative pixels and normalize each channel to the mean of the first
   channel (a first-order correction for the uncalibrated per-channel
   effective areas).
3. Invert the reference frame (``time=15``, the frame the distortion fit was
   optimized against) with :class:`ctis.inverters.MartInverter`,
   starting from a spatially-uniform Gaussian spectral seed.
4. Invert every other frame in a warm-start chain outward from the reference
   frame, reusing the reference frame's weights
   (checked to be :math:`\chi^2`-equivalent to per-frame rebuilds).

The baseline scene grid is measured from the linearized instrument itself:
roughly 0.73 arcsec spatial pixels and 17 km/s velocity bins,
the finest plate scale the instrument delivers in any channel.

**Environment note:** this report currently requires unreleased branches of
two sibling libraries, installed editable:
``optika`` on ``feature/interpolated-system`` (`optika PR #152
<https://github.com/sun-data/optika/pull/152>`_)
and ``ctis`` on ``feature/electron-based-instruments`` (`ctis PR #18
<https://github.com/sun-data/ctis/pull/18>`_).
It is also too expensive for the documentation build:
computing the baseline product needs hundreds of gigabytes of memory,
so this notebook is not part of the documentation toctree yet.

In [ ]:
import numpy as np
import scipy.ndimage
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import esis

## Loading the Level-4 Product

:func:`esis.flights.f1.data.level_4` computes the product on the first call
and caches it in ``~/.esis/cache``;
subsequent calls load the stored result.
The first call is a supercomputer-scale job at the baseline grid
(the regridding weights alone are tens of gigabytes),
so it is normally executed once on a large-memory machine and the cache
carried to wherever the analysis runs.

In [ ]:
%%time
l4 = esis.flights.f1.data.level_4()
l4.outputs.shape

## Convergence and Normalization

The final mean :math:`\chi^2` of every channel, for every frame.
The inversion is stable across the flight;
isolated single-channel spikes mark frames where a particle hit survived
despiking in one camera.
The number of iterations shows the warm-start chain at work:
the reference frame pays the full cost of converging from the seed,
while its neighbors refine an already-converged solution.

In [ ]:
frame = na.arange(0, l4.shape[l4.axis_time], axis=l4.axis_time)

with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        nrows=2,
        sharex=True,
        figsize=(8, 6),
        constrained_layout=True,
    )
    na.plt.plot(frame, l4.mean_chi_squared, ax=axs[0], axis=l4.axis_time)
    axs[0].axhline(1, color="black", linestyle="dotted")
    axs[0].set_ylabel("mean $\chi^2$")
    na.plt.plot(frame, l4.num_iteration, ax=axs[1], axis=l4.axis_time)
    axs[1].set_xlabel("frame")
    axs[1].set_ylabel("iterations")

The channel-normalization factors are stable in time,
and directly measure the uncalibrated relative throughput of the four
channels (up to :math:`\sim 1.5\times`).

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.plot(frame, l4.factor_norm, ax=ax, axis=l4.axis_time)
    ax.set_xlabel("frame")
    ax.set_ylabel("normalization factor")

## Line Intensity Maps

The total reconstructed intensity of each line window for the reference
frame.
The reconstruction recovers solar network structure across the full
(untruncated) octagonal field of view.

In [ ]:
index_reference = 15
labels = ["He I 584", "Mg X 610", "O V 630"]

intensity_reference = l4.intensity[{l4.axis_time: index_reference}]

with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=l4.num_line,
        figsize=(14, 5),
        constrained_layout=True,
        sharex=True,
        sharey=True,
    )
    for i, ax in enumerate(axs.flat):
        map_i = intensity_reference[{l4.axis_line: i}]
        na.plt.pcolormesh(
            l4.inputs.position.x,
            l4.inputs.position.y,
            C=map_i.value,
            ax=ax,
            vmin=0,
            vmax=np.percentile(map_i.value, 99.5),
        )
        ax.set_aspect("equal")
        ax.set_title(labels[i])
        fig.colorbar(
            ax.collections[0],
            ax=ax,
            location="bottom",
            label=f"intensity ({map_i.unit:latex_inline})",
        )

## Full-Field Movies

The evolution of the O V 630 intensity over the flight.
The opacity of the Doppler movie is corrected for the relative atmospheric
transmission of each frame
(the payload observes through more atmosphere toward the ends of the
flight, so the raw signal fades even though the Sun does not).

In [ ]:
index_OV = 2
l4.to_jshtml(l4.animate_intensity(index_line=index_OV))

The Doppler movie encodes the intensity-weighted mean velocity of the
reconstructed O V profile as color — blue toward the observer, red away —
and the line intensity as opacity,
so only regions with significant emission show color.

In [ ]:
l4.to_jshtml(l4.animate_doppler(index_line=index_OV))

## Event E

The 2019 flight captured several compact explosive events;
the largest — labeled *e)* in the Level-3 difference image of
Parker et al. (2022, ApJ 938, 116), there called the "main event" — shows
strong red- and blue-shifted O V emission.
It is located directly in the reconstruction as the brightest feature with a
large intensity-weighted mean Doppler velocity:
the intensity times the absolute mean velocity is smoothed and the maximum
taken inside the octagonal field of view
(center near (+17″, −21″), radius 330″ on the sky).

In [ ]:
intensity_OV = intensity_reference[{l4.axis_line: index_OV}]
velocity_OV = l4.velocity_mean[{l4.axis_line: index_OV, l4.axis_time: index_reference}]

source = (intensity_OV.axes.index(l4.axis_x), intensity_OV.axes.index(l4.axis_y))
intensity_xy = np.moveaxis(intensity_OV.ndarray.value, source, (0, 1))
velocity_xy = np.moveaxis(
    velocity_OV.to(u.km / u.s).ndarray.value,
    (velocity_OV.axes.index(l4.axis_x), velocity_OV.axes.index(l4.axis_y)),
    (0, 1),
)

x_vertices = l4.inputs.position.x.ndarray.to_value(u.arcsec)
y_vertices = l4.inputs.position.y.ndarray.to_value(u.arcsec)
x_center = (x_vertices[:-1] + x_vertices[1:]) / 2
y_center = (y_vertices[:-1] + y_vertices[1:]) / 2

xx, yy = np.meshgrid(x_center, y_center, indexing="ij")
radius = np.hypot(xx - 17, yy + 21)
signature = scipy.ndimage.gaussian_filter(
    np.nan_to_num(intensity_xy * np.abs(velocity_xy)),
    sigma=2,
)
signature[radius > 330] = 0
i_event, j_event = np.unravel_index(np.argmax(signature), signature.shape)
x_event = x_center[i_event]
y_event = y_center[j_event]
x_event, y_event

The Doppler evolution of the event:
a compact blue-red pair at launch develops into a redshifted core with a
blueshifted jet extending north, then fades.

In [ ]:
halfwidth_zoom = 40

fig, ax = plt.subplots(constrained_layout=True)
animation = l4.animate_doppler(index_line=index_OV, ax=ax)
ax.set_xlim(x_event - halfwidth_zoom, x_event + halfwidth_zoom)
ax.set_ylim(y_event - halfwidth_zoom, y_event + halfwidth_zoom)
ax.set_title(f"Event E ({x_event:.0f}\u2033, {y_event:.0f}\u2033)")
l4.to_jshtml(animation)

In [ ]:
fig, ax = plt.subplots(constrained_layout=True)
animation = l4.animate_intensity(index_line=index_OV, ax=ax)
ax.set_xlim(x_event - halfwidth_zoom, x_event + halfwidth_zoom)
ax.set_ylim(y_event - halfwidth_zoom, y_event + halfwidth_zoom)
ax.set_title(f"Event E ({x_event:.0f}\u2033, {y_event:.0f}\u2033)")
l4.to_jshtml(animation)

## Field-Averaged Spectra

The field-averaged spectrum of the reference frame,
one panel per line window,
on a common Doppler-velocity axis.
The measured width of each baseline-subtracted profile is annotated against
the width of the Gaussian seed:
the data reshape the seed rather than being dominated by it,
and the excess width is consistent with the instrumental profile
(the ~18 km/s per pixel dispersion and the velocity bin width)
plus real broadening.

In [ ]:
spectrum = esis.flights.f1.spectrum
width_seed = [
    spectrum.He_I.width_doppler,
    spectrum.Mg_X.width_doppler,
    spectrum.O_V.width_doppler,
]

spectrum_mean = l4.outputs[{l4.axis_time: index_reference}].mean((l4.axis_x, l4.axis_y))
velocity = l4.velocity
velocity_center = l4.velocity_center

# no sharey: sharing the y axis across quantity stairs plots corrupts the
# axis limits (NaN), which makes constrained_layout collapse the panels --
# and the three lines span an order of magnitude in radiance anyway.
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=l4.num_line,
        figsize=(12, 4),
        sharex=True,
        constrained_layout=True,
    )
    for i, ax in enumerate(axs.flat):
        profile_i = spectrum_mean[l4.window(i)]
        profile_sub = profile_i - profile_i.min()
        moment_0 = profile_sub.sum(l4.axis_wavelength)
        v_mean = (profile_sub * velocity_center).sum(l4.axis_wavelength) / moment_0
        sigma = np.sqrt(
            (profile_sub * np.square(velocity_center - v_mean)).sum(l4.axis_wavelength)
            / moment_0
        )
        sigma = sigma.ndarray.to_value(u.km / u.s)
        sigma_seed = width_seed[i].to_value(u.km / u.s)
        na.plt.stairs(
            velocity,
            profile_i,
            ax=ax,
            axis=l4.axis_wavelength,
        )
        ax.text(
            0.03,
            0.97,
            f"$\sigma$ = {sigma:.1f} km/s\nseed $\sigma$ = {sigma_seed:.1f} km/s",
            transform=ax.transAxes,
            ha="left",
            va="top",
        )
        ax.set_title(labels[i])
        ax.set_xlabel(f"velocity ({u.km / u.s:latex_inline})")
        ax.set_ylim(bottom=0)
        if i > 0:
            ax.set_ylabel(None)
    axs.flat[0].set_ylabel(f"mean radiance ({spectrum_mean.unit:latex_inline})")

## Next Steps

- **Point spread function** — the dominant remaining residual is a heavy
  positive tail (signal the forward model cannot reproduce); adding a PSF /
  scattered-light model to the linearized system is the most promising route
  toward :math:`\chi^2 \approx 1` in every channel.
- **Radiometric calibration** — promote the simple mean-matching channel
  normalization to a fitted per-channel scale on the effective area of the
  linear system (or a masked normalization as in Parker et al. 2022).
- **Per-frame pointing** — the product is reconstructed in the reference
  frame's apparent pointing, so the payload drift appears as image motion;
  applying the fitted per-frame pointing requires settling a sign-convention
  discrepancy observed between the fitted pointing deltas and the measured
  image shifts.
- **Gap cells** — exclude the spurious wavelength cells between line windows
  instead of retaining them in the reconstruction.
- **Field-stop mask** — constrain the reconstruction to the interior of the
  octagonal field stop, which MART currently leaks intensity beyond.
- **Validation** — forward-model a synthetic scene
  (:func:`esis.flights.f1.data.synth.scene_iris` or
  :func:`~esis.flights.f1.data.synth.scene_aia`) through the same linear
  system and invert it, so the reconstruction can be compared against a
  known truth.
- **Documentation build** — decide how this report joins the documentation
  toctree, given that the baseline product cannot be computed on the CI
  runner.